# 🎯 Few-Shot Learning

**Teach LLMs by example, not by explanation**

---

## 📋 Overview

**What you'll learn:**
- Zero-shot vs Few-shot vs Many-shot
- Selecting good examples
- Example ordering strategies
- Dynamic few-shot selection
- Production few-shot systems

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
from typing import List, Dict
import random

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🎓 Understanding Few-Shot Learning

### 📚 The Spectrum:

**Zero-Shot** (0 examples)
```
Classify the sentiment: "I love this product!"
```

**Few-Shot** (2-10 examples)
```
Example 1: "Great!" → Positive
Example 2: "Terrible" → Negative

Classify: "I love this product!"
```

**Many-Shot** (10-100+ examples)
```
[100 examples...]

Classify: "I love this product!"
```

### 📊 When to use each:

| Examples | When | Accuracy | Cost |
|----------|------|----------|------|
| **0** | Task is clear | Good | Low |
| **2-5** | Need format/style | Better | Medium |
| **5-10** | Complex patterns | Best | High |
| **10+** | Very specific | Diminishing returns | Very High |

## 🔄 Zero-Shot vs Few-Shot Comparison

In [ ]:
def zero_shot(prompt: str) -> str:
    """Zero-shot: Just ask directly."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=50
    )
    return response.choices[0].message.content

def few_shot(task: str, examples: List[Dict], input_text: str) -> str:
    """Few-shot: Provide examples."""
    
    # Build prompt with examples
    prompt_parts = [task, ""]
    
    for i, ex in enumerate(examples, 1):
        prompt_parts.append(f"Example {i}:")
        prompt_parts.append(f"Input: {ex['input']}")
        prompt_parts.append(f"Output: {ex['output']}")
        prompt_parts.append("")
    
    prompt_parts.append("Now you try:")
    prompt_parts.append(f"Input: {input_text}")
    prompt_parts.append("Output:")
    
    prompt = "\n".join(prompt_parts)
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=50
    )
    return response.choices[0].message.content

# Test: Email subject line generation
email_body = "Hi team, our Q4 revenue increased by 25% compared to last year. Great work!"

print("📧 Task: Generate email subject line\n")
print("Email body:")
print(email_body)
print("\n" + "="*50 + "\n")

# Zero-shot
print("🔸 Zero-Shot:")
result = zero_shot(f"Generate a subject line for this email:\n{email_body}")
print(result)

# Few-shot
examples = [
    {
        "input": "Meeting scheduled for 3pm today to discuss the project timeline.",
        "output": "📅 Project Timeline Meeting Today at 3pm"
    },
    {
        "input": "The system will be down for maintenance from 2am to 4am.",
        "output": "⚠️ Scheduled Maintenance: 2am-4am"
    },
]

print("\n🔹 Few-Shot (with 2 examples):")
result = few_shot(
    "Generate a concise email subject line with an emoji.",
    examples,
    email_body
)
print(result)

print("\n💡 Notice: Few-shot learned the emoji + format pattern!")

## 🎯 Example Selection Strategies

In [ ]:
# Example pool for sentiment classification
SENTIMENT_EXAMPLES = [
    # Positive
    {"text": "Absolutely love it!", "sentiment": "Positive"},
    {"text": "Best purchase ever!", "sentiment": "Positive"},
    {"text": "Exceeded expectations!", "sentiment": "Positive"},
    {"text": "Really happy with this.", "sentiment": "Positive"},
    
    # Negative
    {"text": "Terrible quality.", "sentiment": "Negative"},
    {"text": "Waste of money.", "sentiment": "Negative"},
    {"text": "Very disappointed.", "sentiment": "Negative"},
    {"text": "Would not recommend.", "sentiment": "Negative"},
    
    # Neutral
    {"text": "It's okay.", "sentiment": "Neutral"},
    {"text": "Works as expected.", "sentiment": "Neutral"},
    {"text": "Average product.", "sentiment": "Neutral"},
    {"text": "Nothing special.", "sentiment": "Neutral"},
]

def balanced_selection(examples: List[Dict], n: int = 6) -> List[Dict]:
    """Select balanced examples (equal from each class)."""
    
    # Group by sentiment
    by_sentiment = {}
    for ex in examples:
        sentiment = ex['sentiment']
        if sentiment not in by_sentiment:
            by_sentiment[sentiment] = []
        by_sentiment[sentiment].append(ex)
    
    # Select equal number from each
    per_class = n // len(by_sentiment)
    selected = []
    
    for sentiment, items in by_sentiment.items():
        selected.extend(random.sample(items, min(per_class, len(items))))
    
    return selected

def diverse_selection(examples: List[Dict], n: int = 6) -> List[Dict]:
    """Select diverse examples (different patterns)."""
    # For demo, just random selection
    # In production, use embedding similarity
    return random.sample(examples, min(n, len(examples)))

# Test selection strategies
print("📊 Balanced Selection (2 from each class):\n")
selected = balanced_selection(SENTIMENT_EXAMPLES, n=6)
for ex in selected:
    print(f"  [{ex['sentiment']:8}] {ex['text']}")

print("\n🎲 Random Diverse Selection:\n")
selected = diverse_selection(SENTIMENT_EXAMPLES, n=6)
for ex in selected:
    print(f"  [{ex['sentiment']:8}] {ex['text']}")

## 🔄 Example Ordering Matters

In [ ]:
def test_ordering(examples: List[Dict], test_input: str) -> Dict:
    """Test if example ordering affects results."""
    
    def classify_with_examples(examples_ordered):
        prompt_parts = ["Classify sentiment as Positive, Negative, or Neutral.", ""]
        
        for ex in examples_ordered:
            prompt_parts.append(f"Text: {ex['text']}")
            prompt_parts.append(f"Sentiment: {ex['sentiment']}")
            prompt_parts.append("")
        
        prompt_parts.append(f"Text: {test_input}")
        prompt_parts.append("Sentiment:")
        
        prompt = "\n".join(prompt_parts)
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=10
        )
        return response.choices[0].message.content.strip()
    
    # Test different orderings
    results = {}
    
    # Original order
    results['original'] = classify_with_examples(examples)
    
    # Reversed
    results['reversed'] = classify_with_examples(examples[::-1])
    
    # Random shuffle
    shuffled = examples.copy()
    random.shuffle(shuffled)
    results['shuffled'] = classify_with_examples(shuffled)
    
    return results

# Test
test_examples = [
    {"text": "Love it!", "sentiment": "Positive"},
    {"text": "Hate it!", "sentiment": "Negative"},
    {"text": "It's fine.", "sentiment": "Neutral"},
]

test_input = "This product is amazing!"

print(f"🧪 Testing input: '{test_input}'\n")
results = test_ordering(test_examples, test_input)

print("Results with different orderings:")
for order, result in results.items():
    print(f"  {order:10} → {result}")

# Check if all same
all_same = len(set(results.values())) == 1
print(f"\n{'✅' if all_same else '⚠️'} All results {'identical' if all_same else 'DIFFERENT'}")
if not all_same:
    print("💡 Example ordering can affect results! Use consistent ordering.")

## 🎨 Dynamic Few-Shot Selection

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

class DynamicFewShotSelector:
    """Select most relevant examples based on similarity to input."""
    
    def __init__(self, examples: List[Dict], text_key: str = 'text'):
        """
        Args:
            examples: Pool of examples
            text_key: Key for text field in examples
        """
        self.examples = examples
        self.text_key = text_key
        
        # Create embeddings for all examples
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        texts = [ex[text_key] for ex in examples]
        self.embeddings = self.model.encode(texts)
        
        print(f"✅ Indexed {len(examples)} examples")
    
    def select_similar(
        self,
        query: str,
        n: int = 5,
        diversity: float = 0.5
    ) -> List[Dict]:
        """
        Select most similar examples.
        
        Args:
            query: Input text
            n: Number of examples
            diversity: 0 = most similar, 1 = most diverse
        """
        # Encode query
        query_embedding = self.model.encode([query])[0]
        
        # Calculate similarities
        similarities = np.dot(self.embeddings, query_embedding)
        
        # Get top N most similar
        top_indices = np.argsort(similarities)[-n:][::-1]
        
        selected = [self.examples[i] for i in top_indices]
        
        return selected

# Create selector
selector = DynamicFewShotSelector(SENTIMENT_EXAMPLES)

# Test with different queries
queries = [
    "This is fantastic!",
    "Not good at all.",
    "It's decent."
]

print("\n🎯 Dynamic Few-Shot Selection:\n")

for query in queries:
    print(f"Query: '{query}'")
    selected = selector.select_similar(query, n=3)
    print("  Most similar examples:")
    for ex in selected:
        print(f"    • {ex['text']} → {ex['sentiment']}")
    print()

## 🏗️ Production Few-Shot System

In [ ]:
from dataclasses import dataclass
from typing import Optional, Callable

@dataclass
class FewShotConfig:
    """Configuration for few-shot learning."""
    n_examples: int = 5
    selection_strategy: str = "balanced"  # balanced, diverse, similar
    shuffle_examples: bool = False
    temperature: float = 0.0
    max_tokens: int = 100

class FewShotEngine:
    """Production-ready few-shot learning engine."""
    
    def __init__(
        self,
        client: OpenAI,
        example_pool: List[Dict],
        config: FewShotConfig = None
    ):
        self.client = client
        self.example_pool = example_pool
        self.config = config or FewShotConfig()
        
        # Initialize selector for similarity-based selection
        if self.config.selection_strategy == "similar":
            self.selector = DynamicFewShotSelector(example_pool)
    
    def select_examples(self, query: str = None) -> List[Dict]:
        """Select examples based on strategy."""
        
        if self.config.selection_strategy == "balanced":
            examples = balanced_selection(
                self.example_pool,
                self.config.n_examples
            )
        
        elif self.config.selection_strategy == "diverse":
            examples = diverse_selection(
                self.example_pool,
                self.config.n_examples
            )
        
        elif self.config.selection_strategy == "similar":
            if query is None:
                raise ValueError("Query required for similar selection")
            examples = self.selector.select_similar(
                query,
                self.config.n_examples
            )
        
        else:
            # Random selection
            examples = random.sample(
                self.example_pool,
                min(self.config.n_examples, len(self.example_pool))
            )
        
        if self.config.shuffle_examples:
            random.shuffle(examples)
        
        return examples
    
    def generate(
        self,
        task_description: str,
        input_text: str,
        examples: List[Dict] = None,
        input_key: str = 'text',
        output_key: str = 'sentiment'
    ) -> Dict:
        """Generate prediction with few-shot examples."""
        
        # Select examples if not provided
        if examples is None:
            examples = self.select_examples(input_text)
        
        # Build prompt
        prompt_parts = [task_description, ""]
        
        for i, ex in enumerate(examples, 1):
            prompt_parts.append(f"Example {i}:")
            prompt_parts.append(f"Input: {ex[input_key]}")
            prompt_parts.append(f"Output: {ex[output_key]}")
            prompt_parts.append("")
        
        prompt_parts.append("Now classify:")
        prompt_parts.append(f"Input: {input_text}")
        prompt_parts.append("Output:")
        
        prompt = "\n".join(prompt_parts)
        
        # Call LLM
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens
        )
        
        return {
            'input': input_text,
            'output': response.choices[0].message.content.strip(),
            'examples_used': len(examples),
            'selection_strategy': self.config.selection_strategy,
            'prompt': prompt,
            'tokens': response.usage.total_tokens
        }

# Test production engine
config = FewShotConfig(
    n_examples=3,
    selection_strategy="balanced",
    temperature=0
)

engine = FewShotEngine(client, SENTIMENT_EXAMPLES, config)

# Test
test_texts = [
    "Absolutely amazing product!",
    "Complete waste of money.",
    "It works fine."
]

print("🧪 Testing Few-Shot Engine\n")

for text in test_texts:
    result = engine.generate(
        "Classify sentiment as Positive, Negative, or Neutral.",
        text
    )
    print(f"Input:  {result['input']}")
    print(f"Output: {result['output']}")
    print(f"Tokens: {result['tokens']}")
    print()

## ✅ Summary

### Key Concepts:

1. **🎯 Few-Shot Learning**
   - Teach by example, not explanation
   - 2-10 examples usually optimal
   - Higher accuracy than zero-shot

2. **📊 Selection Strategies**
   - **Balanced**: Equal from each class
   - **Diverse**: Cover different patterns
   - **Similar**: Most relevant to input

3. **🔄 Example Ordering**
   - Can affect results
   - Use consistent ordering
   - Test different orderings

4. **🎨 Dynamic Selection**
   - Choose examples based on input
   - Use embedding similarity
   - Better performance

### Best Practices:

```python
# ✅ Good: Clear format, diverse examples
examples = [
    {"input": "...", "output": "..."},  # Positive
    {"input": "...", "output": "..."},  # Negative
    {"input": "...", "output": "..."},  # Neutral
]

# ❌ Bad: All same class, inconsistent format
examples = [
    "Great! → Positive",
    "Awesome → Positive",
    "Love it! Positive",  # Inconsistent
]
```

### Optimal Number of Examples:

| Task Complexity | Examples | Why |
|----------------|----------|-----|
| Simple (sentiment) | 2-3 | Enough to show pattern |
| Medium (extraction) | 3-5 | Cover edge cases |
| Complex (reasoning) | 5-10 | Show various approaches |
| Very Complex | 10+ | Diminishing returns |

### Cost Optimization:

```python
# Example costs (GPT-3.5-turbo)
0 examples:   50 tokens  → $0.000025
3 examples:  150 tokens  → $0.000075 (3x)
10 examples: 500 tokens  → $0.000250 (10x)

# Use minimum effective examples!
```

### Selection Strategy Guide:

**Use Balanced when:**
- Classification tasks
- Equal class distribution desired
- Avoid bias toward one class

**Use Similar when:**
- Input varies widely
- Context matters
- Need most relevant examples

**Use Diverse when:**
- Want to cover edge cases
- Show different patterns
- General robustness

### Performance Comparison:

```
Task: Email classification

Zero-shot:              75% accuracy
Few-shot (random):      82% accuracy (+7%)
Few-shot (balanced):    85% accuracy (+10%)
Few-shot (similar):     88% accuracy (+13%)
```

### Common Mistakes:

- ❌ Too many examples (diminishing returns)
- ❌ Inconsistent formatting
- ❌ All examples from same class
- ❌ Examples too similar to each other
- ❌ Not testing different strategies

### Next: `03_prompt_engineering/07_prompt_optimization.ipynb`